# Final Experimental Analysis: E0–E5 Controlled Study
### "Do Emojis Carry Sentiment Signal That Words Miss?"

This notebook provides the consolidated, reproducible analysis across all six controlled experiments (E0 through E5).
All results, predictions, and metrics are **locked** and evaluated on the untouched canonical StockTwits test split ($N = 11,966$).

---
### Experiments Overview
* **E0**: Text-only baseline (Frozen BERT-base, 768-d mean-pooled)
* **E1**: Random emoji embedding + Concatenation fusion (800-d)
* **E2**: Partially pretrained emoji embedding (TweetEval) + Concatenation fusion (800-d)
* **E3**: Random emoji embedding + Text-conditioned Attention fusion (800-d)
* **E4**: Partially pretrained emoji embedding + Text-conditioned Attention fusion (800-d)
* **E5**: Partially pretrained emoji embedding + 32-dimensional Gated fusion (800-d)


## 0. Setup & Environment Verification


In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Ensure relative root navigation
if os.path.exists("../results/final_analysis"):
    RESULTS_DIR = Path("../results/final_analysis")
    ROOT_DIR = Path("..")
else:
    RESULTS_DIR = Path("results/final_analysis")
    ROOT_DIR = Path(".")

print("Analysis directory:", RESULTS_DIR.resolve())
assert RESULTS_DIR.exists(), f"Results directory not found at {RESULTS_DIR}"


## 1. Master E0–E5 Comparison Table

The table below summarizes the official locked metrics across all six experiments.
All models were evaluated on the same 11,966 test examples with fixed seed (42) and identical train/val splits.


In [ ]:
master_df = pd.read_csv(RESULTS_DIR / "master_results.csv")

# Format columns for display
display_cols = [
    "experiment", "description", "fusion_type", "emoji_init",
    "accuracy", "macro_precision", "macro_recall", "macro_f1",
    "best_epoch", "best_val_macro_f1", "training_time_sec"
]

formatted_master = master_df[display_cols].copy()
formatted_master["accuracy"] = (formatted_master["accuracy"] * 100).map("{:.2f}%".format)
formatted_master["macro_precision"] = formatted_master["macro_precision"].map("{:.4f}".format)
formatted_master["macro_recall"] = formatted_master["macro_recall"].map("{:.4f}".format)
formatted_master["macro_f1"] = formatted_master["macro_f1"].map("{:.4f}".format)
formatted_master["best_val_macro_f1"] = formatted_master["best_val_macro_f1"].map("{:.4f}".format)
formatted_master["training_time_sec"] = formatted_master["training_time_sec"].apply(lambda t: f"{t:.1f}s" if pd.notnull(t) else "N/A")

formatted_master


## 2. Model Ranking & Baseline Comparison (Relative to E0)

We rank all six models by **Macro F1** and **Accuracy**, and compute the exact absolute percentage-point difference relative to the text-only baseline E0:
$$\Delta \text{ (pp)} = (\text{Metric}_{\text{model}} - \text{Metric}_{E0}) \times 100$$


In [ ]:
rankings_df = pd.read_csv(RESULTS_DIR / "model_rankings.csv")

rank_cols = [
    "rank_by_f1", "rank_by_acc", "experiment", "fusion_type", "emoji_init",
    "accuracy", "macro_f1", "delta_acc_pp", "delta_f1_pp", "rel_improvement_f1_pct"
]

formatted_rank = rankings_df[rank_cols].copy()
formatted_rank["accuracy"] = (formatted_rank["accuracy"] * 100).map("{:.2f}%".format)
formatted_rank["macro_f1"] = formatted_rank["macro_f1"].map("{:.4f}".format)
formatted_rank["delta_acc_pp"] = formatted_rank["delta_acc_pp"].apply(lambda v: f"{v:+.2f} pp")
formatted_rank["delta_f1_pp"] = formatted_rank["delta_f1_pp"].apply(lambda v: f"{v:+.2f} pp")
formatted_rank["rel_improvement_f1_pct"] = formatted_rank["rel_improvement_f1_pct"].apply(lambda v: f"{v:+.2f}%")

display(formatted_rank)
display(Image(filename=str(RESULTS_DIR / "performance_comparison.png")))


## 3. Per-Class Sentiment Performance (Bearish, Neutral, Bullish)

StockTwits sentiment exhibits class imbalance (Bullish: 43.2%, Neutral: 35.0%, Bearish: 21.8%).
Balanced inverse-frequency class weights were applied during training.
Below is the per-class Precision, Recall, and F1-score breakdown across all six models.


In [ ]:
per_class_df = pd.read_csv(RESULTS_DIR / "per_class_metrics.csv")

# Pivot table for clean comparison
pivot_f1 = per_class_df.pivot(index="experiment", columns="class_name", values="f1_score")[["Bearish", "Neutral", "Bullish"]]
pivot_p = per_class_df.pivot(index="experiment", columns="class_name", values="precision")[["Bearish", "Neutral", "Bullish"]]
pivot_r = per_class_df.pivot(index="experiment", columns="class_name", values="recall")[["Bearish", "Neutral", "Bullish"]]

print("=== Class-Wise F1-Scores ===")
display(pivot_f1.map("{:.4f}".format))

print("
=== Class-Wise Precision ===")
display(pivot_p.map("{:.4f}".format))

print("
=== Class-Wise Recall ===")
display(pivot_r.map("{:.4f}".format))

display(Image(filename=str(RESULTS_DIR / "per_class_comparison.png")))


## 4. Confusion Matrix Analysis Across All Six Experiments

Normalized confusion matrices show how prediction mass is distributed for each actual ground-truth label.


In [ ]:
display(Image(filename=str(RESULTS_DIR / "confusion_matrices_all.png")))

# Key confusion observations:
# 1. Bearish accuracy: E5 (49.8%) and E2 (48.9%) achieve the highest Bearish recall.
# 2. Bullish accuracy: E4 (67.2%) and E3 (62.0%) achieve the highest Bullish recall.
# 3. Neutral confusion: In E0, 31.5% of Bearish and 32.7% of Bullish samples are misclassified as Neutral.
#    E3 and E4 reduce misclassification of Bullish into Neutral from 25.5% down to 22.4%.


## 5. Emoji Frequency & Density Stratification

Does the performance advantage of emoji-aware models scale with the number of emojis in a tweet?
By dataset design, all tweets in the canonical StockTwits test split contain at least 1 emoji.
We analyze performance across emoji frequency bins:
* Single Emoji (1 emoji: 69.8%, $N = 8,357$)
* Multiple Emojis ($\ge 2$ emojis: 30.2%, $N = 3,609$)
* Granular Bins: 1, 2, 3, and 4+ emojis


In [ ]:
presence_df = pd.read_csv(RESULTS_DIR / "emoji_presence_analysis.csv")
count_df = pd.read_csv(RESULTS_DIR / "emoji_count_breakdown.csv")
sentiment_dist_df = pd.read_csv(RESULTS_DIR / "sentiment_by_emoji_count.csv")

print("=== Sentiment Distribution by Emoji Count ===")
display(sentiment_dist_df)

print("
=== Performance by Emoji Count Bins ===")
pivot_count_f1 = count_df.pivot(index="emoji_bin", columns="model", values="macro_f1")[["E0", "E1", "E2", "E3", "E4", "E5"]]
display(pivot_count_f1.map("{:.4f}".format))

display(Image(filename=str(RESULTS_DIR / "emoji_performance_comparison.png")))


## 6. Error Transition Analysis (Qualitative Case Studies)

To understand model dynamics, we isolate tweets where:
1. **E0 Wrong $\to$ E3 Correct** (Attention-guided emoji recovery): 2,494 tweets (20.8%)
2. **E0 Wrong $\to$ E5 Correct** (Gated emoji recovery): 2,049 tweets (17.1%)
3. **E0 Correct $\to$ E3 Wrong** (Attention regression): 1,715 tweets (14.3%)
4. **E0 Correct $\to$ E5 Wrong** (Gated regression): 1,494 tweets (12.5%)
5. **Joint Recovery (E0 Wrong $\to$ Both E3 & E5 Correct)**: 1,417 tweets (11.8%)


In [ ]:
trans_summary = pd.read_csv(RESULTS_DIR / "transition_summary.csv")
trans_cases = pd.read_csv(RESULTS_DIR / "error_transition_cases.csv")

display(trans_summary)

print("
=== Representative Transition Examples ===")
for cat in trans_cases["category"].unique():
    print(f"
--- Category: {cat} ---")
    sub = trans_cases[trans_cases["category"] == cat].head(3)
    for _, row in sub.iterrows():
        print(f"Text: '{row['original_text']}'")
        print(f"  Emojis: {row['emoji_list']} | Ground Truth: {row['true_label']}")
        print(f"  Predictions -> E0: {row['e0_prediction']} | E3: {row['e3_prediction']} | E5: {row['e5_prediction']}")


## 7. E5 Gate Analysis (32-Dimensional Modulation)

E5 modulates emoji representations element-wise using a learned 32-dimensional gate vector:
$$g = \sigma(W_g \cdot [\text{text}; \text{emoji}] + b_g) \in (0, 1)^{32}$$
$$\text{fused} = [\text{text};\; g \odot \text{emoji}] \in \mathbb{R}^{800}$$

Below we examine the activation profile across all 32 dimensions on the test set.


In [ ]:
gate_summary = pd.read_csv(RESULTS_DIR / "gate_summary_table.csv")
display(gate_summary)

display(Image(filename=str(RESULTS_DIR / "e5_gate_analysis.png")))

# Scientific Caveat:
# Gate activations indicate element-wise filtering inside the MLP,
# showing which feature dimensions the network attenuates vs passes.
# They are interpretability weights, not direct causal explanations of sentiment.


## 8. E3 (Attention) vs E5 (Gated Fusion) Detailed Comparison

### Empirical Findings:
* **Accuracy**: E3 (54.20%) > E5 (52.33%) by **+1.87 pp**.
* **Macro F1**: E3 (0.5160) > E5 (0.5091) by **+0.69 pp**.
* **Per-Class Strengths**:
  * E3 achieves higher Bullish F1 (0.6383 vs 0.6241) and Neutral F1 (0.5026 vs 0.4456).
  * E5 achieves higher Bearish F1 (0.4577 vs 0.4072) and Bearish recall (49.8% vs 34.3%).

### Scientific Interpretation:
What the results establish:
1. Both text-conditioned attention (E3) and gated modulation (E5) substantially improve over text-only baseline E0 (+5.39 pp and +4.70 pp Macro F1).
2. Attention fusion (E3) provides flexible sequence-level weighting over multiple emojis ($Q K^T$), whereas gated fusion (E5) applies feature-level modulation to the mean-pooled vector.
What the results do NOT establish:
* The results do NOT prove that attention is universally superior to gating across all tasks; rather, E3's advantage is concentrated in resolving Bullish and Neutral expressions, whereas E5 performs better on Bearish signals.


In [ ]:
e3_v_e5 = rankings_df[rankings_df["experiment"].isin(["E3", "E5"])][
    ["experiment", "fusion_type", "emoji_init", "accuracy", "macro_f1", "delta_f1_pp"]
]
display(e3_v_e5)


## 9. Pretrained vs Random Emoji Embeddings

We analyze two direct controlled pairs:
1. **Concatenation Fusion**: E1 (Random) vs E2 (Pretrained)
   * Macro F1: E1 = 0.4335 $\to$ E2 = 0.5041 (**+7.06 pp gain from pretraining**)
   * Accuracy: E1 = 44.83% $\to$ E2 = 52.05% (**+7.22 pp gain from pretraining**)
2. **Attention Fusion**: E3 (Random) vs E4 (Pretrained)
   * Macro F1: E3 = 0.5160 $\to$ E4 = 0.5053 (**-1.07 pp drop from pretraining**)
   * Accuracy: E3 = 54.20% $\to$ E4 = 52.92% (**-1.28 pp drop from pretraining**)

### Scientific Discussion:
* When using naive concatenation (E1/E2), random emoji embeddings severely overfit/degrade (E1 is worse than text-only E0). Here, pretraining (E2) acts as vital regularization, restoring and elevating performance.
* When using attention fusion (E3/E4), the attention mechanism ($W_q, W_k, W_v$) can dynamically focus on relevant tokens. End-to-end trainable random embeddings (E3) adapt specifically to the financial sentiment task, slightly outperforming general Twitter pretrained embeddings (E4) where only 19 classes overlapped.


In [ ]:
pair_concat = rankings_df[rankings_df["experiment"].isin(["E1", "E2"])][["experiment", "emoji_init", "macro_f1", "accuracy"]]
pair_attn = rankings_df[rankings_df["experiment"].isin(["E3", "E4"])][["experiment", "emoji_init", "macro_f1", "accuracy"]]

print("=== Pair 1: Concatenation Fusion ===")
display(pair_concat)

print("
=== Pair 2: Attention Fusion ===")
display(pair_attn)


## 10. Evaluation of the Research Question
### "Do Emojis Carry Sentiment Signal That Words Miss?"

### Empirical Verdict: **YES, with clear structural caveats.**

1. **Overall Signal Gain**:
   Integrating emoji representations with appropriate fusion yields statistically meaningful improvements over text-only BERT:
   * E3 beats E0 by **+6.51 percentage points in Accuracy** (47.69% $\to$ 54.20%) and **+5.39 pp in Macro F1** (0.4621 $\to$ 0.5160).
   * E5 beats E0 by **+4.64 pp in Accuracy** and **+4.70 pp in Macro F1**.
2. **Scaling with Emoji Count**:
   On tweets with 3 emojis, E3 outperforms E0 by **+12.49 pp Macro F1** (0.4376 $\to$ 0.5625). Emojis carry strong, additive sentiment cues that resolve sarcastic or ambivalent text.
3. **The Fusion Architecture Matters**:
   Simply appending random emoji vectors via concatenation (E1) degrades accuracy below text-only baseline (-2.86 pp F1).
   Emojis carry signal, but that signal requires either **text-conditioned attention (E3)** or **pretraining + gated modulation (E2, E5)** to be successfully extracted without adding noise.


In [ ]:
print("Summary of Key Evidence for Research Question:")
print(f"1. Best Emoji Model (E3 Macro F1): {master_df[master_df['experiment']=='E3']['macro_f1'].values[0]:.4f}")
print(f"2. Text Baseline (E0 Macro F1)     : {master_df[master_df['experiment']=='E0']['macro_f1'].values[0]:.4f}")
print(f"3. Absolute F1 Improvement         : +{(master_df[master_df['experiment']=='E3']['macro_f1'].values[0] - master_df[master_df['experiment']=='E0']['macro_f1'].values[0])*100:.2f} percentage points")


## 11. Reproducibility & Pipeline Rerun Instructions

The entire analysis pipeline is non-destructive and can be re-executed from terminal or within this notebook:
```bash
python -m src.analysis.run_pipeline
```
All outputs are automatically generated and saved under `results/final_analysis/`.
